# 08 · The `device_exposure` table

Medical devices the patient has or is given: implants (stents, pacemakers), equipment and
supplies (catheters, crutches, CPAP). Each device gets a required `kind` label:

| Label | Means | Visit / provider |
|---|---|---|
| `placed` | implanted, inserted or supplied in this encounter | from the note |
| `existing` | the patient already has or uses it | left empty |

Unlike `drug_exposure`, the end date is optional and never inferred. `quantity` is the
number in the note, else 1.

Details: [the device_exposure table](https://precisiondata.github.io/aatm/latest/getting-started/omop_tables/#the-device_exposure-table).

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. Mapper and notes

The same mapper as in notebook 04 codes the extracted names.

In [2]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline
from aatm.selectors import OpenAILLMSelector
from aatm.terminology_mapper import TerminologyMapper

mapper = TerminologyMapper(
    retriever="text-embedding-3-small", selector=OpenAILLMSelector("gpt-5-mini")
)
mapper.retriever.top_k = 30

notes = [
    ClinicalNote(
        text=(
            "s/p PCI with 2 drug-eluting stents in 2021. Dual-chamber pacemaker since 2019.\n"
            "Foley catheter inserted today for retention. Walks with crutches.\n"
            "No indwelling lines. Candidate for ICD. Nicotine patch 21 mg daily."
        ),
        person_id=40,
        note_date="2025-06-01",
        visit_occurrence_id=950,
        provider_id=14,
    )
]

## 2. Run

In [3]:
pipeline = OmopExtractionPipeline(
    "device_exposure", model_id="openai/gpt-5-mini", coder=mapper
)
tables = await pipeline(notes)

tables["device_exposure"][
    [
        "device_source_value",
        "device_concept_id",
        "device_exposure_start_date",
        "quantity",
        "visit_occurrence_id",
        "provider_id",
    ]
]

,device_source_value,device_concept_id,device_exposure_start_date,quantity,visit_occurrence_id,provider_id
0,drug-eluting stents,4131681,2021-01-01,2,<NA>,<NA>
1,Dual-chamber pacemaker,45767339,2019-01-01,1,<NA>,<NA>
2,Foley catheter,40625746,2025-06-01,1,950,14
3,crutches,4251933,2025-06-01,1,<NA>,<NA>


Not rows: the PCI is a procedure (only the stents are devices), "no indwelling lines" is
negated, the ICD is only considered, and the nicotine patch is a drug (`drug_exposure`).

## 3. Check the coded concepts and the issues

Device names are often short ("CPAP", "brace"), and right answers can still score low, e.g. "crutches" → *Crutch* at 0.59 here. Review them before setting a `min_score`.

In [4]:
tables.mappings_dataframe()[
    ["row_id", "column", "text", "concept_id", "concept_name", "domain_id", "score"]
]

,row_id,column,text,concept_id,concept_name,domain_id,score
0,1,device_concept_id,drug-eluting stents,4131681,Drug eluting stent,Device,0.791424
1,2,device_concept_id,Dual-chamber pacemaker,45767339,Dual chamber demand implantable cardiac pacemaker,Device,0.634333
2,3,device_concept_id,Foley catheter,40625746,Foley catheter,Device,0.999998
3,4,device_concept_id,crutches,4251933,Crutch,Device,0.588283


In [5]:
tables.issues_dataframe()[["row_id", "column", "reason", "detail"]]

,row_id,column,reason,detail
0,1,device_exposure_start_date,imputed_date_parts,'2021' has no day or month; read as 2021-01-01
1,2,device_exposure_start_date,imputed_date_parts,'2019' has no day or month; read as 2019-01-01


## Next

Write your own definition for another table: see notebook 03 and the [`omop` block reference](https://precisiondata.github.io/aatm/latest/getting-started/omop_tables/#reference-the-omop-block).